# Predicting Customer Dissatisfaction Before It's Posted

*Binary classification on 99k Brazilian e-commerce orders (Olist).*

Notebook 1 of the planned series, covering the EDA. The goal is; understand the data well enough to know which features are worth building, where to draw the line between a good and a bad review (split), and which factors a business could actually fix. Modelling is not the only goal.

The analysis runs in five passes: check the shape, check the quality, look at each feature on its own, look at each feature against the target, then look at the features together. 

This covers univariate, bivariate as well as multivariate analysis.

## Problem statement

The goal is to predict whether a delivered order will get a bad review, and to identify which factors drive it.

The prediction happens at one specific moment: the order has just been delivered and the customer has not written a review yet. Anything created after that moment is off limits, because the model would not have it when it needs to make the call. 

That rules out variables such as the review text/comments and both review timestamps.

Two outputs come from the same analysis:

1. A predicted probability for each delivered order, which ranks orders by how likely they are to get a bad review.
2. A ranked list of which factors drive bad reviews, and which of those a business can actually control.

The second output is the more useful of the two. A probability tells you which orders look risky. 

One row is one order(grain). The target is `review_score`, collapsed into good and bad. The cut-off is not set yet. It gets decided later in this notebook, once the score distribution and its relationship to the features are clear.

## Modeling population

The modeling population is delivered orders only.

An order that was cancelled or never arrived is close to guaranteed a bad review. The data confirms this further down: 71% of failed orders got one star. A model trained on those rows learns that cancelled means bad, scores well, and tells the business nothing it did not already know.

A second reason matters more. Those rows are easy to predict, so keeping them would push accuracy up without the model getting any better at the cases that count. The metrics would be flattering rather than useful.

The harder question is the interesting one: why did an order that actually arrived still get rated badly? A team can act on that.

Two things get checked before the filter is applied, both further down:

- How much data the filter costs. Dropping rows is only cheap when there are few of them.
- Whether failed orders really are almost all bad reviews. If so, they carry no useful signal and are safe to remove.

Both checks support the filter, so it is applied in the cleaning step.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
DATA_DIR = "../data/raw/"

## Understanding the data

The data arrives as eight separate CSV files. Each one gets checked for row and column counts, missing values, and how it joins to the others.

### Load the tables

In [ ]:

df_customers = pd.read_csv(DATA_DIR + "olist_customers_dataset.csv")
df_geolocation = pd.read_csv(DATA_DIR + "olist_geolocation_dataset.csv")
df_order_items = pd.read_csv(DATA_DIR + "olist_order_items_dataset.csv")
df_order_payments = pd.read_csv(DATA_DIR + "olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv(DATA_DIR + "olist_order_reviews_dataset.csv")
df_orders = pd.read_csv(DATA_DIR + "olist_orders_dataset.csv")
df_products = pd.read_csv(DATA_DIR + "olist_products_dataset.csv")
df_sellers = pd.read_csv(DATA_DIR + "olist_sellers_dataset.csv")

### Size and missing values

One table comparing all eight, rather than eight separate summaries.

In [ ]:
tables = {
    "customers": df_customers,
    "geolocation": df_geolocation,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "orders": df_orders,
    "products": df_products,
    "sellers": df_sellers,
}

profile = []

for name in tables:
    df = tables[name]
    profile.append(
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "null_cells": df.isnull().sum().sum(),
            "columns_with_nulls": (df.isnull().sum() > 0).sum(),
        }
    )

profile = pd.DataFrame(profile)
profile

,table,rows,columns,null_cells,columns_with_nulls
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,145903,2
5,orders,99441,8,4908,3
6,products,32951,9,2448,8
7,sellers,3095,4,0,0


Three things stand out.

`geolocation` is by far the largest table at one million rows, and it maps zip codes to coordinates rather than holding anything tied to an order.

`order_reviews` has the most missing data by a wide margin, but all of it sits in the two comment columns. The review score itself is complete, which is what matters since it is the target.

`orders` has missing values in three of its date columns. These are most likely orders that never finished, and the missing values section covers that.

### Preview the rows

A first look at the columns in each table, before working out how they join.

In [ ]:
display("Orders:", df_orders.head(3))
display("Order Items:", df_order_items.head(3))
display("Order Payments:", df_order_payments.head(3))
display("Order Reviews:", df_order_reviews.head(3))
display("Customers:", df_customers.head(3))
display("Sellers:", df_sellers.head(3))
display("Products:", df_products.head(3))

'Orders:'

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


'Order Items:'

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


'Order Payments:'

...

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


'Sellers:'

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


'Products:'

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


### How the tables connect (a guess, not yet checked)

This join map is a guess, built from shared column names and the row count gaps. Nothing here is verified yet. That is the job of the quality checks below.

- orders to customers: `customer_id`, one to one.
- orders to order_items: `order_id`, probably one to many, since order_items has more rows (112,650) than orders (99,441).
- orders to order_payments: `order_id`, probably one to many as well. An order can be paid in more than one way.
- orders to order_reviews: `order_id`. The row counts are close but not equal, so counts alone settle nothing here.
- order_items to products: `product_id`.
- order_items to sellers: `seller_id`.

The `geolocation` table is left out. It maps zip codes to coordinates, and the location detail this project needs is already in the customers and sellers tables as city and state. Leaving it out keeps the merge simpler.

### Converting the date columns

The five date columns in `orders` load as text, so no date maths is possible on them yet. They get converted here, once, so the ordering check below works and the feature work later does not have to repeat it.

The missing counts are printed before and after. A value that fails to convert becomes a null, so a jump in the null count would mean some dates use a format pandas did not expect.

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print("missing before conversion:")

print(df_orders[date_cols].isnull().sum())

for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

print('')

print("missing after conversion:")

print(df_orders[date_cols].isnull().sum())

print('')

print("date range:", df_orders["order_purchase_timestamp"].min(), "to", df_orders["order_purchase_timestamp"].max())

missing before conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

missing after conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


The counts match exactly before and after, so every value parsed correctly and no dates use an unexpected format. The missing values were already missing in the raw file.

Purchases run from September 2016 to October 2018, which is a little over two years. That has two consequences worth noting now.

The first is that seasonality is worth checking. Two years is enough to see whether order volume and review scores move with the time of year.

The second matters more. Because the data covers a long stretch of time, splitting it randomly into training and test sets would put later orders into the training set and earlier orders into the test set. The model would effectively be learning from the future to predict the past, which inflates its scores and would not be possible in real use. 

The train test split can be time based instead, and the section below checks whether the data is stable enough for that to work.

The missing dates line up with the pattern expected from unfinished orders: 160 never approved, 1,783 never handed to a carrier, 2,965 never delivered. Each stage has more missing values than the one before it. 

The missing values will also check and confirms this with `order_status`.

### Do the timestamps run in the right order?

Each order moves through four dates in a fixed order: purchased, approved, handed to the carrier, delivered. A fifth date, the estimated delivery date, is set at purchase.

Timestamps outside of this out of order can looks valid but is not. 

therefore the three checks catch impossible sequences before any feature gets built on top of them.

In [ ]:
bad_approved = (df_orders["order_approved_at"] < df_orders["order_purchase_timestamp"]).sum()
bad_carrier = (df_orders["order_delivered_carrier_date"] < df_orders["order_approved_at"]).sum()
bad_delivered = (df_orders["order_delivered_customer_date"] < df_orders["order_delivered_carrier_date"]).sum()

print("approved before purchase:", bad_approved)
print("handed to carrier before approved:", bad_carrier)
print("delivered before handed to carrier:", bad_delivered)

approved before purchase: 0
handed to carrier before approved: 1359
delivered before handed to carrier: 23


Two of the three checks found problems.

Zero orders were approved before they were purchased, so the start of the sequence is sound.
1,359 orders were handed to the carrier before the payment was approved this is fine since ; `order_approved_at` records when the payment system confirmed, and for some payment types confirmation can take time and meanwhile the seller may ship; like Brazilian boleto payments are a bank slip rather than an instant card charge, so a delay there is possible. 

The problem there is that any feature measuring the gap between approval and carrier pickup will be negative for these 1,359 orders, must get handled before theappropriate feature construction takes place.

23 orders were delivered before they were handed to the carrier. That has no sensible explanation. A package cannot arrive before it ships, so these are broken records. At 23 rows out of 99,441 they are small enough to drop, and they get removed in the cleaning step.

## Data quality checks

Three checks, each answering a different question. 

Are there duplicate rows? 

Is each table's key unique? 

And do the keys in one table actually exist in the table they point to?

In [ ]:
for name in tables:
    print(name, "duplicate rows:", tables[name].duplicated().sum())

customers duplicate rows: 0
geolocation duplicate rows: 261831
order_items duplicate rows: 0
order_payments duplicate rows: 0
order_reviews duplicate rows: 0
orders duplicate rows: 0
products duplicate rows: 0
sellers duplicate rows: 0


This check looks for rows identical across every column, not just a repeated key. It catches a file loaded twice, which a key check on its own would miss.

Zero duplicate rows across all eight tables.

In [ ]:
print("customer_id in customers:", df_customers["customer_id"].is_unique)
print("order_id in orders:", df_orders["order_id"].is_unique)
print("product_id in products:", df_products["product_id"].is_unique)
print("seller_id in sellers:", df_sellers["seller_id"].is_unique)
print("order_id in order_reviews:", df_order_reviews["order_id"].is_unique)
print("order_id in order_items:", df_order_items["order_id"].is_unique)
print("order_id in order_payments:", df_order_payments["order_id"].is_unique)

customer_id in customers: True
order_id in orders: True
product_id in products: True
seller_id in sellers: True
order_id in order_reviews: False
order_id in order_items: False
order_id in order_payments: False


`customers`, `orders`, `products`, and `sellers` each have a unique key, so they merge onto orders directly without changing the row count.

`order_items` and `order_payments` both have repeated `order_id` values, confirming the one to many guess. cant merge onto orders directly without breaking the our grain: 1 row per order, so both get aggregated to order level first.

`order_id` in `order_reviews` is not unique either, which is unexpected. Reviews should be one per order to make the predictions. This affects the target, so it must be checked too

In [ ]:
print("orders.customer_id exists in customers:", df_orders["customer_id"].isin(df_customers["customer_id"]).all())
print("order_items.order_id exists in orders:", df_order_items["order_id"].isin(df_orders["order_id"]).all())
print("order_payments.order_id exists in orders:", df_order_payments["order_id"].isin(df_orders["order_id"]).all())
print("order_reviews.order_id exists in orders:", df_order_reviews["order_id"].isin(df_orders["order_id"]).all())
print("order_items.product_id exists in products:", df_order_items["product_id"].isin(df_products["product_id"]).all())
print("order_items.seller_id exists in orders:", df_order_items["seller_id"].isin(df_sellers["seller_id"]).all())

orders.customer_id exists in customers: True
order_items.order_id exists in orders: True
order_payments.order_id exists in orders: True
order_reviews.order_id exists in orders: True
order_items.product_id exists in products: True
order_items.seller_id exists in orders: True


All six come back True. Every key that points from one table to another lands on a real row, with no orphans.

This matters because I am about to merge with `how="left"`. A left join hides broken keys as nulls rather than raising an error, so without this check I would find out about a problem later and mistake it for missing data.

Combined with the uniqueness results above, the join map is now verified rather than guessed.

### Checking the target: orders and order_reviews

`review_score` is what I am predicting, so I check this join before the others. If reviews are missing for some orders I lose training rows, and if they are duplicated for others I count the same order twice. Both are worse than getting a feature slightly wrong.

In [ ]:
print("rows in order_reviews:", len(df_order_reviews))
print("unique order_id in order_reviews:", df_order_reviews["order_id"].nunique())
print("unique order_id in orders:", df_orders["order_id"].nunique())
print("duplicated order_id in order_reviews:", df_order_reviews["order_id"].duplicated().sum())
print("\nmissing values in order_reviews:")
print(df_order_reviews.isnull().sum())

rows in order_reviews: 99224
unique order_id in order_reviews: 98673
unique order_id in orders: 99441
duplicated order_id in order_reviews: 551

missing values in order_reviews:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


Two problems, both small.

99,441 orders exist but only 98,673 of them have a review, so 768 orders have no score at all. I cannot train on those and they get dropped in the cleaning step.

551 orders have more than one review row. `review_score` itself is never missing, so the target is complete for every order that has a review. The two comment columns are mostly empty, which is expected. Most people leave a star rating and dont bother to leave a comment

Before I decide what to do about the duplicates, i must check

### Do the duplicate reviews agree?

551 orders have more than one review. I plan to keep one review per order.

If both reviews give the same score, the choice is harmless anything. If they give different scores, then I am choosing the label myself, and that is worth knowing about since the label is what the whole model is trained on.

In [ ]:
dupe_mask = df_order_reviews["order_id"].duplicated(keep=False)
dupe_reviews = df_order_reviews[dupe_mask]

# for each affected order, how many different scores did it receive? 
scores_per_order = dupe_reviews.groupby("order_id")["review_score"].nunique()

print("orders with more than one review:", len(scores_per_order))
print("number of different scores given per order:")
print(scores_per_order.value_counts())

orders with more than one review: 547

number of different scores given per order:
review_score
1    345
2    202
Name: count, dtype: int64


345 had only 1 unique value meaning review_score for these are the same during dedupe we can choose any method for this wont cause any problems

202 values are ambiguous since they had 2 different review scores

In [ ]:
# 1. Filter dataset to orders that have conflicting review scores
disagreeing_order_ids = scores_per_order[scores_per_order > 1].index
disagreeing_reviews = dupe_reviews[dupe_reviews["order_id"].isin(disagreeing_order_ids)]

# 2. Get the lowest and highest score for each conflicting order
review_summary = disagreeing_reviews.groupby("order_id")["review_score"].agg(
    lowest_score="min",
    highest_score="max"
)

# 3. Calculate how far apart the ratings are (e.g., 5-star vs 1-star = gap of 4)
review_summary["score_gap"] = review_summary["highest_score"] - review_summary["lowest_score"]

print("Frequency of rating gaps between reviews for the same order:")
print(review_summary["score_gap"].value_counts().sort_index())

# 4. Check if the reviews conflict on sentiment (Good: 4–5 stars and Bad: 1–3 stars)
lowest_is_good = review_summary["lowest_score"] >= 4
highest_is_good = review_summary["highest_score"] >= 4

# Count orders where one review is "Good" and another is "Bad"
sentiment_conflicts = (lowest_is_good != highest_is_good).sum()

print("Orders where reviews flip overall sentiment (Good vs Bad):", sentiment_conflicts)

Frequency of rating gaps between reviews for the same order:
score_gap
1    90
2    47
3    32
4    33
Name: count, dtype: int64
Orders where reviews flip overall sentiment (Good vs Bad): 118


Of the 547 orders with more than one review, 345 received the same score twice and 202 received different scores.

Looking at how far apart the disagreements are: 90 differ by one star, and 33 differ by four stars, meaning the same order received both a 1 and a 5. That is a wide gap for a single purchase.

The number that matters is the last one. Under the good and bad split used later in this notebook, 118 of these orders would land in a different class depending on which review is kept. For those orders the label is not being read from the data, it is being chosen here.

118 orders out of roughly 96,000 is around 0.1% of the modeling population, so this is not large enough to affect the model

**Decision: keep the most recent review per order.** because domain understanding is The later review is the customer's final position.

Dropping all 547 orders loses valid data for the 345 that agree.

### How many customers order more than once?

In [ ]:
print("rows in customers:", len(df_customers))
print("unique people:", df_customers["customer_unique_id"].nunique())

rows in customers: 99441
unique people: 96096


Almost every customer appears only once. There is no repeat customer behavior here to be learnt from the model

It also means I do not have to worry about the same customer appearing in both the training and test sets, which would otherwise let the model learn a specific person's patterns rather than a general pattern. Model learns general patterns that apply for unseen buyers too.

### Table relationships (verified)

![Table relationships](assets/table_relationships.png)

Green edges are one to one or many to one, so those tables merge onto orders directly. Orange edges are one to many, so `order_items` and `order_payments` get summarised to order level first.